# 📊 Analyse des Accidents de la Route (2012-2019)

**Projet** : Mighty Mosquitoes - Data Engineering 2025

**Base de données** : `accidents_db`

**Objectif** : Analyser les données d'accidents pour identifier les zones à risque, les tendances temporelles, et les conditions accidentogènes.

---

## 📋 Table des matières

1. [Configuration et connexion](#1-configuration-et-connexion)
2. [Section 1 : Zones à risque](#2-section-1--zones-à-risque)
3. [Section 2 : Conditions de survenue](#3-section-2--conditions-de-survenue)
4. [Section 3 : Tendances temporelles](#4-section-3--tendances-temporelles)
5. [Section 4 : Conditions à risque](#5-section-4--conditions-à-risque)
6. [Section 5 : Zones fréquentées](#6-section-5--zones-fréquentées)
7. [Section 6 : Anomalies temporelles](#7-section-6--anomalies-temporelles)
8. [Export des résultats](#8-export-des-résultats)

---

## 1. Configuration et connexion

### Installation des dépendances

Si ce n'est pas déjà fait, installer les packages nécessaires :

```bash
pip install pandas sqlalchemy psycopg2-binary plotly matplotlib seaborn python-dotenv
```

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# Import du SQL Loader (DRY - source unique de vérité)
from sql_loader import SQLQueryLoader

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configuration Matplotlib
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Packages importés avec succès")

In [ ]:
# 🔄 Rechargement automatique des modules (utile pour le développement)
# Permet de modifier mock_data.py sans redémarrer le kernel
%load_ext autoreload
%autoreload 2

print("✅ Autoreload activé - Les modifications de mock_data.py seront automatiquement prises en compte")

In [ ]:
# ============================================================
# CONFIGURATION : MODE DE DONNÉES
# ============================================================
# True  = Utiliser données mockées (pour tester visualisations sans DB)
# False = Utiliser base de données réelle via sql_loader
USE_MOCK_DATA = True

if USE_MOCK_DATA:
    from mock_data import generate_mock_data
    MOCK_DATA = generate_mock_data()
    print()
    print("⚠️  MODE MOCK ACTIVÉ")
    print("   → Données factices chargées (pour tests/présentation)")
    print("   → Pour utiliser la vraie DB : USE_MOCK_DATA = False")
else:
    from sql_loader import SQLQueryLoader
    loader = SQLQueryLoader()
    print()
    print("✅ MODE PRODUCTION")
    print("   → Connexion à la base de données requise")
    print("   → Requêtes chargées depuis etl/sql/")
    print()
    print("📖 Documentation:")
    print("   - Catalogue requêtes : ../etl/sql/query_catalog.md")
    print("   - Guide visualisations : ../etl/sql/guide_visualisations.md")

### Palette de couleurs du projet

In [ ]:
# Palette Mighty Mosquitoes
PALETTE = {
    'danger': '#D32F2F',      # Rouge (tués, danger)
    'warning': '#F57C00',     # Orange (hospitalisés)
    'caution': '#FBC02D',     # Jaune (blessés légers)
    'safe': '#388E3C',        # Vert (indemnes)
    'neutral': '#757575',     # Gris (neutre)
    'primary': '#1976D2',     # Bleu (principal)
    'secondary': '#512DA8',   # Violet (secondaire)
}

print("✅ Palette de couleurs définie")

### Connexion à la base de données

**Option 1** : Connexion directe (dev)

In [ ]:
# Paramètres de connexion
DB_CONFIG = {
    'user': 'your_username',
    'password': 'your_password',
    'host': 'localhost',
    'port': 5432,
    'database': 'accidents_db'
}

# Création de l'engine SQLAlchemy
connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
engine = create_engine(connection_string)

# Test de connexion
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT COUNT(*) FROM analytics.accident_usagers_aggr"))
        nb_accidents = result.scalar()
    print(f"✅ Connexion réussie : {nb_accidents:,} accidents détectés")
except Exception as e:
    print(f"❌ Erreur de connexion : {e}")

**Option 2** : Connexion avec `.env` (prod - recommandé)

In [ ]:
# from dotenv import load_dotenv
# import os
#
# load_dotenv()
#
# connection_string = os.getenv('DATABASE_URL')
# engine = create_engine(connection_string)
#
# print("✅ Connexion via .env établie")

### Fonction utilitaire : Exécution de requêtes

In [ ]:
def executer_requete(query, engine=engine, verbose=True):
    """
    Exécute une requête SQL et retourne un DataFrame.
    
    Args:
        query (str): Requête SQL à exécuter
        engine: SQLAlchemy engine
        verbose (bool): Afficher les infos de succès
    
    Returns:
        pd.DataFrame: Résultats de la requête
    """
    try:
        start_time = datetime.now()
        df = pd.read_sql(query, engine)
        duration = (datetime.now() - start_time).total_seconds()
        
        if verbose:
            print(f"✅ Requête exécutée en {duration:.2f}s - {len(df):,} lignes retournées")
        
        return df
    except Exception as e:
        print(f"❌ Erreur : {e}")
        return None

print("✅ Fonction executer_requete() définie")

In [ ]:
# Section 1.1 : Top 10 régions
# Requête source : section1_kpi_*.sql
# Documentation : ../etl/sql/query_catalog.md#11

if USE_MOCK_DATA:
    df_1_1 = MOCK_DATA['1_1']
    print(f"📊 Données mockées chargées ({{len(df_1_1):,}} lignes)")
else:
    query_1_1 = loader.load_query('section1', '1.1')
    df_1_1 = pd.read_sql(query_1_1, engine)
    print(f"✅ Requête 1.1 exécutée - {{len(df_1_1):,}} lignes")

df_1_1.head()

#### Visualisation : Barres horizontales

In [ ]:
fig = px.bar(
    df_1_1,
    y='reg_name',
    x='nb_accidents',
    orientation='h',
    title='Top 10 Régions - Nombre d\'accidents (2012-2019)',
    labels={'reg_name': 'Région', 'nb_accidents': 'Nombre d\'accidents'},
    color='nb_accidents',
    color_continuous_scale='Reds',
    text='nb_accidents',
    height=500
)

fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

### 1.2 - Top 15 départements (taux de gravité)

In [ ]:
# Section 1.2 : Top 15 départements (gravité)
# Requête source : section1_kpi_*.sql
# Documentation : ../etl/sql/query_catalog.md#12

if USE_MOCK_DATA:
    df_1_2 = MOCK_DATA['1_2']
    print(f"📊 Données mockées chargées ({{len(df_1_2):,}} lignes)")
else:
    query_1_2 = loader.load_query('section1', '1.2')
    df_1_2 = pd.read_sql(query_1_2, engine)
    print(f"✅ Requête 1.2 exécutée - {{len(df_1_2):,}} lignes")

df_1_2.head()

#### Visualisation : Barres empilées

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    y=df_1_2['dep_name'], x=df_1_2['tues'],
    name='Tués', orientation='h',
    marker_color=PALETTE['danger'],
    text=df_1_2['tues'], textposition='inside'
))

fig.add_trace(go.Bar(
    y=df_1_2['dep_name'], x=df_1_2['hosp'],
    name='Hospitalisés', orientation='h',
    marker_color=PALETTE['warning'],
    text=df_1_2['hosp'], textposition='inside'
))

fig.add_trace(go.Bar(
    y=df_1_2['dep_name'], x=df_1_2['legers'],
    name='Blessés légers', orientation='h',
    marker_color=PALETTE['caution'],
    text=df_1_2['legers'], textposition='inside'
))

fig.update_layout(
    barmode='stack',
    title='Top 15 Départements - Répartition de la gravité',
    yaxis={'categoryorder': 'total ascending'},
    xaxis_title='Nombre de victimes',
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

---

## 3. Section 2 : Conditions de survenue

### 2.1 - Répartition par agglomération

In [ ]:
# Section 2.1 : Répartition agglomération
# Requête source : section2_kpi_*.sql
# Documentation : ../etl/sql/query_catalog.md#21

if USE_MOCK_DATA:
    df_2_1 = MOCK_DATA['2_1']
    print(f"📊 Données mockées chargées ({{len(df_2_1):,}} lignes)")
else:
    query_2_1 = loader.load_query('section2', '2.1')
    df_2_1 = pd.read_sql(query_2_1, engine)
    print(f"✅ Requête 2.1 exécutée - {{len(df_2_1):,}} lignes")

df_2_1.head()

#### Visualisation : Donut chart

In [ ]:
fig = px.pie(
    df_2_1,
    names='agglomeration',
    values='nb_accidents',
    title='Répartition des accidents par type d\'agglomération',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Accidents: %{value:,.0f}<br>Pourcentage: %{percent}'
)

fig.show()

### 2.2 - Heatmap (Luminosité × Atmosphère)

In [ ]:
# Section 2.2 : Heatmap Luminosité × Atmosphère (requête custom)
# Note : Requête custom (croisement 2D non présent dans section*.sql)

if USE_MOCK_DATA:
    df_2_2 = MOCK_DATA['2_2']
    print("📊 Données mockées chargées")
else:
    # Requête custom (croisement Luminosité × Atmosphère)
    query_2_2 = """
SELECT
    lum.libelle as luminosite,
    atm.libelle as atmosphere,
    COUNT(*) as nb_accidents
FROM analytics.accident_usagers_aggr a
JOIN analytics.dim_lum lum ON a.lum = lum.lum
JOIN analytics.dim_atm atm ON a.atm = atm.atm
GROUP BY lum.libelle, atm.libelle
"""
    df_2_2 = pd.read_sql(query_2_2, engine)
    print(f"✅ Requête exécutée - {{len(df_2_2):,}} lignes")

# Pivot pour heatmap
pivot_2_2 = df_2_2.pivot(index='luminosite', columns='atmosphere', values='nb_accidents')
pivot_2_2.head()

#### Visualisation : Heatmap annotée

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot_2_2,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Nombre d\'accidents'},
    ax=ax
)
ax.set_title('Matrice Luminosité × Conditions atmosphériques', fontweight='bold', fontsize=14)
ax.set_xlabel('Conditions atmosphériques', fontweight='bold')
ax.set_ylabel('Luminosité', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## 4. Section 3 : Tendances temporelles

### 3.1 - Évolution annuelle

In [ ]:
# Section 3.1 : Évolution annuelle
# Requête source : section3_kpi_*.sql
# Documentation : ../etl/sql/query_catalog.md#31

if USE_MOCK_DATA:
    df_3_1 = MOCK_DATA['3_1']
    print(f"📊 Données mockées chargées ({{len(df_3_1):,}} lignes)")
else:
    query_3_1 = loader.load_query('section3', '3.1')
    df_3_1 = pd.read_sql(query_3_1, engine)
    print(f"✅ Requête 3.1 exécutée - {{len(df_3_1):,}} lignes")

df_3_1.head()

#### Visualisation : Line chart avec tendance

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_3_1['annee'],
    y=df_3_1['nb_accidents'],
    mode='lines+markers',
    name='Accidents',
    line=dict(color=PALETTE['primary'], width=3),
    marker=dict(size=10)
))

fig.add_trace(go.Scatter(
    x=df_3_1['annee'],
    y=df_3_1['tendance'],
    mode='lines',
    name='Tendance',
    line=dict(color=PALETTE['neutral'], width=2, dash='dash')
))

fig.update_layout(
    title='Évolution annuelle des accidents (2012-2019)',
    xaxis_title='Année',
    yaxis_title='Nombre d\'accidents',
    hovermode='x unified',
    height=500
)

fig.show()

### 3.2 - Heatmap jour × heure

In [ ]:
# Section 3.2 : Heatmap jour × heure (requête custom)
# Note : Requête custom (croisement jour × heure)

if USE_MOCK_DATA:
    df_3_2 = MOCK_DATA['3_2']
    print("📊 Données mockées chargées")
else:
    # Requête custom (heatmap jour × heure)
    query_3_2 = """
SELECT
    jour,
    EXTRACT(HOUR FROM hrmn::time) as heure,
    COUNT(*) as nb_accidents
FROM analytics.accident_usagers_aggr
WHERE jour IS NOT NULL AND hrmn IS NOT NULL
GROUP BY jour, EXTRACT(HOUR FROM hrmn::time)
"""
    df_3_2 = pd.read_sql(query_3_2, engine)
    print(f"✅ Requête exécutée - {len(df_3_2):,} lignes")

# Noms des jours
jours_noms = {1: 'Lundi', 2: 'Mardi', 3: 'Mercredi', 4: 'Jeudi',
              5: 'Vendredi', 6: 'Samedi', 7: 'Dimanche'}
df_3_2['jour_nom'] = df_3_2['jour'].map(jours_noms)

# Pivot
pivot_3_2 = df_3_2.pivot(index='jour_nom', columns='heure', values='nb_accidents')

# Réordonner les jours
ordre_jours = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
pivot_3_2 = pivot_3_2.reindex(ordre_jours)

pivot_3_2.head()

#### Visualisation : Heatmap calendaire

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    pivot_3_2,
    cmap='YlOrRd',
    annot=False,
    cbar_kws={'label': 'Nombre d\'accidents'},
    linewidths=0.5,
    ax=ax
)
ax.set_title('Calendrier hebdomadaire des accidents (jour × heure)', fontweight='bold', fontsize=14)
ax.set_xlabel('Heure de la journée', fontweight='bold')
ax.set_ylabel('Jour de la semaine', fontweight='bold')
plt.tight_layout()
plt.show()

---

## 5. Section 4 : Conditions à risque

### 4.1 - Quadrant de risque (Fréquence × Gravité)

In [ ]:
# Section 4.1 : Quadrant de risque
# Requête source : section4_kpi_*.sql
# Documentation : ../etl/sql/query_catalog.md#41

if USE_MOCK_DATA:
    df_4_1 = MOCK_DATA['4_1']
    print(f"📊 Données mockées chargées ({{len(df_4_1):,}} lignes)")
else:
    query_4_1 = loader.load_query('section4', '4.1')
    df_4_1 = pd.read_sql(query_4_1, engine)
    print(f"✅ Requête 4.1 exécutée - {{len(df_4_1):,}} lignes")

df_4_1.head()

#### Visualisation : Scatter plot avec quadrants

In [ ]:
# Calculer les médianes pour les quadrants
mediane_freq = df_4_1['frequence'].median()
mediane_grav = df_4_1['gravite_moyenne'].median()

fig = px.scatter(
    df_4_1,
    x='frequence',
    y='gravite_moyenne',
    text='condition',
    size='frequence',
    color='gravite_moyenne',
    title='Quadrant de risque : Fréquence × Gravité moyenne',
    labels={
        'frequence': 'Fréquence (nombre d\'accidents)',
        'gravite_moyenne': 'Gravité moyenne (tués par accident)'
    },
    color_continuous_scale='RdYlGn_r',
    height=600
)

# Ajouter les lignes de quadrant
fig.add_hline(y=mediane_grav, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=mediane_freq, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_traces(textposition='top center')
fig.show()

---

## 6. Section 5 : Zones fréquentées

### 5.1 - Top 20 communes

In [ ]:
# Section 5.1 : Top 20 communes
# Requête source : section5_kpi_*.sql
# Documentation : ../etl/sql/query_catalog.md#51

if USE_MOCK_DATA:
    df_5_1 = MOCK_DATA['5_1']
    print(f"📊 Données mockées chargées ({{len(df_5_1):,}} lignes)")
else:
    query_5_1 = loader.load_query('section5', '5.1')
    df_5_1 = pd.read_sql(query_5_1, engine)
    print(f"✅ Requête 5.1 exécutée - {{len(df_5_1):,}} lignes")

df_5_1.head()

#### Visualisation : Scatter plot

In [ ]:
fig = px.scatter(
    df_5_1,
    x='nb_accidents',
    y='nb_tues',
    size='nb_accidents',
    color='nb_tues',
    hover_name='com_name',
    hover_data={'dep_name': True, 'nb_accidents': ':,.0f', 'nb_tues': ':,.0f'},
    text='com_name',
    title='Top 20 communes accidentogènes (Accidents × Tués)',
    labels={'nb_accidents': 'Nombre d\'accidents', 'nb_tues': 'Nombre de tués'},
    color_continuous_scale='Reds',
    height=700
)

fig.update_traces(textposition='top center', textfont_size=8)
fig.show()

---

## 7. Section 6 : Anomalies temporelles

### 6.1 - Z-score temporel (détection d'anomalies)

In [ ]:
# Section 6.1 : Z-score temporel
# Requête source : section6_kpi_*.sql
# Documentation : ../etl/sql/query_catalog.md#61

if USE_MOCK_DATA:
    df_6_1 = MOCK_DATA['6_1']
    print(f"📊 Données mockées chargées ({{len(df_6_1):,}} lignes)")
else:
    query_6_1 = loader.load_query('section6', '6.1')
    df_6_1 = pd.read_sql(query_6_1, engine)
    print(f"✅ Requête 6.1 exécutée - {{len(df_6_1):,}} lignes")

df_6_1.head()

#### Visualisation : Time series avec bandes de confiance

In [ ]:
fig = go.Figure()

# Série principale
fig.add_trace(go.Scatter(
    x=df_6_1['date'],
    y=df_6_1['nb_accidents'],
    mode='lines',
    name='Accidents mensuels',
    line=dict(color=PALETTE['primary'], width=2)
))

# Bande de confiance (moyenne ± 2σ)
fig.add_trace(go.Scatter(
    x=df_6_1['date'],
    y=df_6_1['moyenne'] + 2*df_6_1['ecart_type'],
    mode='lines',
    name='Limite supérieure (μ+2σ)',
    line=dict(width=0),
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=df_6_1['date'],
    y=df_6_1['moyenne'] - 2*df_6_1['ecart_type'],
    mode='lines',
    name='Limite inférieure (μ-2σ)',
    fill='tonexty',
    fillcolor='rgba(128,128,128,0.2)',
    line=dict(width=0),
    showlegend=True
))

# Marquer les anomalies
anomalies = df_6_1[df_6_1['anomalie']]
fig.add_trace(go.Scatter(
    x=anomalies['date'],
    y=anomalies['nb_accidents'],
    mode='markers',
    name='Anomalies (|z| > 2)',
    marker=dict(size=12, color=PALETTE['danger'], symbol='x')
))

fig.update_layout(
    title='Détection d\'anomalies temporelles (Z-score)',
    xaxis_title='Date',
    yaxis_title='Nombre d\'accidents',
    hovermode='x unified',
    height=600
)

fig.show()